# 03 - Baseline: TF-IDF + Logistic Regression (Phase 7)
Pipeline: `text_tfidf  ->  TF-IDF (unigrams / unigrams+bigrams)  ->  Logistic Regression  ->  prediction`

* settings chosen on the **validation** set, test set used once
* class-weighted (`class_weight="balanced"`)
* MACD Telugu + Kannada are scored as a **zero-shot** check (never trained on)
* IndicGuard is **not** used here

Run all cells. The full grid search can take a few minutes on the real data.
Saved: `results/metrics/baseline_metrics.json`, `results/figures/baseline_confusion_matrix.png`.

> Section 6 prints real comments / words that can be abusive. **Clear all outputs before committing.**

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid")

from src.utils.config import (TRAIN_FILE, VAL_FILE, TEST_FILE, ZERO_SHOT_FILE, EXTERNAL_FILE,
                              FIGURES_DIR, METRICS_DIR, LABEL_NAMES)
from src.data.io_utils import read_processed_csv
from src.data.preprocess import clean_for_xlmr, clean_for_tfidf
FIGURES_DIR.mkdir(parents=True, exist_ok=True); METRICS_DIR.mkdir(parents=True, exist_ok=True)

from IPython.display import Image, display
from src.baseline.train_baseline import run_baseline, top_features

## 2. Train, tune and evaluate
`QUICK = True` runs a single setting (about 1 minute) - use it to test. Use `False` for the real result.

In [ ]:
QUICK = False
res = run_baseline(quick=QUICK, verbose=True)

## 3. Validation grid (how we chose unigrams vs bigrams and C)

In [ ]:
grid = pd.read_csv(METRICS_DIR / "baseline_val_grid.csv")
display(grid.sort_values("val_macro_f1", ascending=False).round(4))
print("Chosen:", res["best_config"])

## 4. Final numbers (validation / test / zero-shot)

In [ ]:
keep = ["n", "accuracy", "precision_hate", "recall_hate", "f1_hate", "macro_f1", "roc_auc"]
overall = pd.DataFrame({k: res[k]["overall"] for k in ["validation", "test", "zero_shot"]}).T[keep].round(4)
display(overall)
print("\nTest set by language:")
display(pd.DataFrame(res["test"]["by_language"]).T[keep].round(4))
print("Test set by source:")
display(pd.DataFrame(res["test"]["by_source"]).T[keep].round(4))
print("Zero-shot (languages the model never saw):")
display(pd.DataFrame(res["zero_shot"]["by_language"]).T[keep].round(4))

## 5. Confusion matrix (test set)

In [ ]:
display(Image(filename=str(FIGURES_DIR / "baseline_confusion_matrix.png")))

## 6. What did the model learn? (top n-grams) and typical mistakes
These are the n-grams with the largest positive / negative weight. Real abusive words will appear - this is expected.

In [ ]:
feats = top_features(res["_objects"]["vectorizer"], res["_objects"]["model"], n=15)
print("Most indicative of HATE/ABUSIVE :", feats["hate"])
print("Most indicative of NON-HATE     :", feats["non_hate"])

pred = pd.read_csv(ROOT / "results" / "predictions" / "baseline_predictions.csv")
t = pred[pred["set"] == "test"]
for name, mask in [("FALSE POSITIVES (non-hate predicted as hate)", (t.label == 0) & (t.pred == 1)),
                   ("FALSE NEGATIVES (hate predicted as non-hate)", (t.label == 1) & (t.pred == 0))]:
    print(f"\n=== {name}: {int(mask.sum())} ===")
    for _, r in t[mask].sample(min(6, int(mask.sum())), random_state=42).iterrows():
        print(f"  [{r.language}] {str(r.text)[:110]}")

## 7. What to write in your report
> *The proposed XLM-R model was compared against a conventional TF-IDF + Logistic Regression baseline
> (word unigrams/bigrams, class-weighted, hyper-parameters chosen on the validation set).*

Copy the test-set macro-F1 per language from section 4: these are the numbers XLM-R has to beat in Phase 8.